# 3.2 动量类指标

## 学习目标
- 掌握 RSI（相对强弱指数）的计算与超买超卖判断
- 理解随机指标 Stochastic（KDJ）
- 理解动量（Momentum）的概念

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

data = yf.download('AAPL', start='2022-01-01', end='2024-01-01', progress=False)
close = data['Close'].squeeze()
high = data['High'].squeeze()
low = data['Low'].squeeze()

## 1. RSI — 相对强弱指数

$$RSI = 100 - \frac{100}{1 + RS}, \quad RS = \frac{\text{n日平均上涨幅度}}{\text{n日平均下跌幅度}}$$

- $RSI > 70$：超买（可能回调）
- $RSI < 30$：超卖（可能反弹）
- 中性区间：30~70

In [ ]:
def compute_rsi(price, period=14):
    delta = price.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(com=period - 1, min_periods=period).mean()
    avg_loss = loss.ewm(com=period - 1, min_periods=period).mean()
    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

rsi = compute_rsi(close)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

ax1.plot(close.index, close.values, linewidth=1.2, color='steelblue')
ax1.set_title('AAPL 收盘价', fontsize=12)
ax1.set_ylabel('价格')
ax1.grid(alpha=0.3)

ax2.plot(rsi.index, rsi.values, linewidth=1.2, color='purple')
ax2.axhline(70, color='red', linestyle='--', alpha=0.7, label='超买 (70)')
ax2.axhline(30, color='green', linestyle='--', alpha=0.7, label='超卖 (30)')
ax2.axhline(50, color='gray', linestyle='-', alpha=0.3)
ax2.fill_between(rsi.index, 70, rsi.values, where=(rsi.values > 70), alpha=0.2, color='red')
ax2.fill_between(rsi.index, 30, rsi.values, where=(rsi.values < 30), alpha=0.2, color='green')
ax2.set_ylim(0, 100)
ax2.set_title('RSI (14)', fontsize=12)
ax2.set_ylabel('RSI')
ax2.legend(loc='upper left')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Stochastic Oscillator (KDJ)

$$K = \frac{P_{close} - L_{n}}{H_{n} - L_{n}} \times 100$$
$$D = EMA_3(K)$$
$$J = 3K - 2D$$

In [ ]:
def compute_kdj(high, low, close, n=9, m=3):
    lowest_low = low.rolling(n).min()
    highest_high = high.rolling(n).max()
    rsv = (close - lowest_low) / (highest_high - lowest_low) * 100
    K = rsv.ewm(com=m - 1, adjust=False).mean()
    D = K.ewm(com=m - 1, adjust=False).mean()
    J = 3 * K - 2 * D
    return K, D, J

K, D, J = compute_kdj(high, low, close)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

ax1.plot(close.index, close.values, linewidth=1.2, color='steelblue')
ax1.set_title('AAPL 收盘价', fontsize=12)
ax1.grid(alpha=0.3)

ax2.plot(K.index, K.values, label='K', linewidth=1.2, color='blue')
ax2.plot(D.index, D.values, label='D', linewidth=1.2, color='orange')
ax2.plot(J.index, J.values, label='J', linewidth=1, color='green', alpha=0.7)
ax2.axhline(80, color='red', linestyle='--', alpha=0.5, label='超买 (80)')
ax2.axhline(20, color='green', linestyle='--', alpha=0.5, label='超卖 (20)')
ax2.set_ylim(-10, 110)
ax2.set_title('KDJ 指标 (9, 3, 3)', fontsize=12)
ax2.legend(loc='upper left', ncol=2)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 3. 动量指标 (Momentum)

$$Mom_n(t) = P_t - P_{t-n}$$

或用收益率表示：$Mom_n(t) = \frac{P_t}{P_{t-n}} - 1$

In [ ]:
mom_20 = close.pct_change(20)   # 20日动量
mom_60 = close.pct_change(60)   # 60日动量

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

ax1.plot(close.index, close.values, linewidth=1.2, color='steelblue')
ax1.set_title('AAPL 收盘价', fontsize=12)
ax1.grid(alpha=0.3)

ax2.plot(mom_20.index, mom_20.values, label='20日动量', linewidth=1.2, color='blue')
ax2.plot(mom_60.index, mom_60.values, label='60日动量', linewidth=1.5, color='red', alpha=0.7)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.fill_between(mom_20.index, 0, mom_20.values,
                 where=(mom_20.values >= 0), alpha=0.15, color='green')
ax2.fill_between(mom_20.index, 0, mom_20.values,
                 where=(mom_20.values < 0), alpha=0.15, color='red')
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax2.set_title('价格动量', fontsize=12)
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 🎯 练习

1. RSI 的超买/超卖阈值通常是 70/30，试改为 80/20，比较信号数量差异。
2. 在 2022 年初上涨行情中，RSI 是否长期处于高位？这说明了什么？
3. 实现规则：RSI < 30 时买入，RSI > 70 时卖出，计算其历史表现。

---
**下一节** → `03_volatility_indicators.ipynb`